# Классификация ботов — quickstart

Ноутбук показывает, как загрузить данные, собрать пару простейших признаков и получить
валидный `submission.csv`. Это **не** решение задачи: скор такого baseline будет чуть выше
константы. Дальше — ваша работа.

Условие и описание метрики — в `README.md`.

In [63]:
import numpy as np
import pandas as pd
import re
import math
from collections import Counter
from pathlib import Path
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from feature_engineering import calculate_entropy, add_ua_features, add_temporal_features
from catboost import CatBoostClassifier

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [64]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 328905 entries, 0 to 328904
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   cookie_id      328905 non-null  str           
 1   event_ts       328905 non-null  datetime64[us]
 2   eid            328905 non-null  int64         
 3   event_name     328905 non-null  str           
 4   platform       328905 non-null  str           
 5   user_agent     328905 non-null  str           
 6   item_id        214308 non-null  float64       
 7   item_category  295985 non-null  str           
 8   item_location  305112 non-null  str           
 9   seller_type    195052 non-null  str           
 10  search_query   100402 non-null  str           
 11  search_page    100402 non-null  float64       
 12  pointer_x      108540 non-null  float64       
 13  pointer_y      108540 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(8)
memory usage

In [65]:
events['platform'] = events['platform'].str.lower().astype('category')
events['item_category'] =  events['item_category'].fillna('unknown').astype('category')
events['item_location'] = events['item_location'].fillna('unknown').astype('category')
events['seller_type'] = events['seller_type'].fillna('unknown').astype('category')
events['event_name'] = events['event_name'].astype('category')
q = events['search_query'].fillna('').astype('string')
events['query_entropy']   = q.apply(calculate_entropy)

## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [66]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
web        138792
android    126875
desktop     46866
ios         12269
iphone       4103
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.000
item_location    0.000
seller_type      0.000
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
query_entropy    0.000
dtype: float64


## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [67]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


In [68]:
def extract_all_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    meta = meta.copy()
    meta['cookie_age_days'] = (meta['window_start_ts'] - meta['cookie_created_at']).dt.total_seconds() / 86400.0

    ev = ev.sort_values(['cookie_id', 'event_ts'])
    ev['time_diff'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
    ev['time_diff_rounded'] = ev['time_diff'].round()
    ev['gap_le_1s'] = ev['time_diff'].le(1).astype(float)
    ev['gap_le_5s'] = ev['time_diff'].le(5).astype(float)
    ev['gap_le_10s'] = ev['time_diff'].le(10).astype(float)
    ev['gap_le_30s'] = ev['time_diff'].le(30).astype(float)
    ev['gap_gt_30m'] = ev['time_diff'].gt(1800).astype(float)

    agg_spec = {
        'eid': 'count',
        'item_id': 'nunique',
        'item_category': 'nunique',
        'item_location': 'nunique',
        'search_query': 'nunique',
        'search_page': ['max', 'mean'],
        'entropy': ['mean', 'min', 'max'],
        'is_known_tool': 'max',
        'is_headless': 'max',
        'has_url': 'max',
        'starts_with_mozilla': 'mean',
        'is_app_header': 'max',
        'ua_device_conflict': 'max',
        'ua_length': ['mean', 'std'],
        'digit_ratio': 'mean',
        'chrome_major_version': ['min', 'max', 'nunique'],
        'time_diff': ['mean', 'std', 'min', 'median'],
        'gap_le_1s': 'mean',
        'gap_le_5s': 'mean',
        'gap_le_10s': 'mean',
        'gap_le_30s': 'mean',
        'gap_gt_30m': ['mean', 'sum'],
        'pointer_x': lambda x: x.notna().mean(),
        'hour_sin_1': 'mean',
        'hour_cos_1': 'mean',
        'hour_sin_2': 'mean',
        'hour_cos_2': 'mean',
        'hour_distance_to_23': ['mean', 'min', 'std'],
        'is_night': 'mean',
        'is_evening': 'mean',
        'is_late_evening': 'mean',
    }

    features = ev.groupby('cookie_id').agg(agg_spec)
    features.columns = ['_'.join(c).strip('_') for c in features.columns]

    # Устойчивые характеристики распределения пауз и регулярности действий.
    gaps = ev.dropna(subset=['time_diff']).copy()
    gap_features = gaps.groupby('cookie_id').agg(
        time_diff_q10=('time_diff', lambda s: s.quantile(0.10)),
        time_diff_q25=('time_diff', lambda s: s.quantile(0.25)),
        time_diff_q75=('time_diff', lambda s: s.quantile(0.75)),
        time_diff_q90=('time_diff', lambda s: s.quantile(0.90)),
        time_diff_unique=('time_diff_rounded', 'nunique'),
        time_diff_count=('time_diff_rounded', 'size'),
    )
    gap_features['time_diff_iqr'] = gap_features.time_diff_q75 - gap_features.time_diff_q25
    gap_features['time_diff_repeat_ratio'] = 1 - gap_features.time_diff_unique / gap_features.time_diff_count.clip(lower=1)
    gap_features['time_diff_cv'] = features.time_diff_std / features.time_diff_mean.clip(lower=1e-6)
    gap_features['time_diff_burstiness'] = (features.time_diff_std - features.time_diff_mean) / (features.time_diff_std + features.time_diff_mean).clip(lower=1e-6)
    gap_features['session_count'] = 1 + features.gap_gt_30m_sum

    event_freq = pd.crosstab(ev['cookie_id'], ev['event_name'], normalize='index')
    event_freq.columns = [f'freq_{c}' for c in event_freq.columns]

    # Полный профиль активности по часам и разности этого профиля с лагами 1–3 часа.
    hour_profile = pd.crosstab(ev['cookie_id'], ev['event_hour'], normalize='index')
    hour_profile = hour_profile.reindex(columns=range(24), fill_value=0)
    hour_profile.columns = [f'hour_share_{h:02d}' for h in hour_profile.columns]
    hour_values = hour_profile.to_numpy()
    hour_lags = {}
    for lag in (1, 2, 3):
        # Разность с предыдущим часом; индекс часов циклический.
        diff = hour_values - np.roll(hour_values, lag, axis=1)
        for h in range(24):
            hour_lags[f'hour_diff_lag{lag}_{h:02d}'] = diff[:, h]
    hour_lags = pd.DataFrame(hour_lags, index=hour_profile.index)
    hour_lag_summary = pd.DataFrame(index=hour_profile.index)
    for lag in (1, 2, 3):
        diff = hour_values - np.roll(hour_values, lag, axis=1)
        hour_lag_summary[f'hour_diff_lag{lag}_mean_abs'] = np.abs(diff).mean(axis=1)
        hour_lag_summary[f'hour_diff_lag{lag}_max_abs'] = np.abs(diff).max(axis=1)

    hour_summary = pd.DataFrame(index=hour_profile.index)
    hour_summary['active_hour_count'] = (hour_values > 0).sum(axis=1)
    hour_summary['hour_share_max'] = hour_values.max(axis=1)
    hour_summary['hour_share_top3'] = np.sort(hour_values, axis=1)[:, -3:].sum(axis=1)
    hour_summary['hour_entropy'] = -(hour_values * np.log(hour_values + 1e-12)).sum(axis=1)
    hour_summary['hour_uniform_l1'] = np.abs(hour_values - 1 / 24).sum(axis=1)
    hour_summary['hour_profile_roughness'] = np.abs(hour_values - np.roll(hour_values, 1, axis=1)).mean(axis=1)
    hour_summary['hour_linear_slope'] = ((hour_values - hour_values.mean(axis=1, keepdims=True)) * (np.arange(24) - 11.5)).sum(axis=1) / ((np.arange(24) - 11.5) ** 2).sum()
    hour_summary['night_share_00_05'] = hour_values[:, 0:6].sum(axis=1)
    hour_summary['morning_share_06_11'] = hour_values[:, 6:12].sum(axis=1)
    hour_summary['day_share_12_17'] = hour_values[:, 12:18].sum(axis=1)
    hour_summary['evening_share_18_23'] = hour_values[:, 18:24].sum(axis=1)
    hour_summary['late_minus_early_share'] = hour_values[:, 18:24].sum(axis=1) - hour_values[:, 0:6].sum(axis=1)
    hour_profile = hour_profile.join(hour_lags).join(hour_lag_summary).join(hour_summary)

    # Строковые представления для CatBoost: mode-категории и текстовые поля.
    def mode_or_missing(s):
        s = s.astype('string').fillna('__MISSING__')
        mode = s.mode()
        return str(mode.iloc[0]) if len(mode) else '__MISSING__'

    categorical_agg = ev.groupby('cookie_id').agg({
        'platform': mode_or_missing,
        'event_name': mode_or_missing,
        'item_category': mode_or_missing,
        'item_location': mode_or_missing,
        'seller_type': mode_or_missing,
        'user_agent': mode_or_missing,
    }).rename(columns={
        'platform': 'cat_platform_mode',
        'event_name': 'cat_event_name_mode',
        'item_category': 'cat_item_category_mode',
        'item_location': 'cat_item_location_mode',
        'seller_type': 'cat_seller_type_mode',
        'user_agent': 'text_user_agent',
    })

    def query_text(s):
        values = s.dropna().astype(str).drop_duplicates().head(100)
        return ' '.join(values) if len(values) else '__EMPTY_QUERY__'

    query_agg = ev.groupby('cookie_id')['search_query'].agg(query_text).rename('text_search_queries')

    df_out = meta[['cookie_id', 'cookie_age_days']].merge(features.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(gap_features.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(event_freq.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(hour_profile.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(categorical_agg.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(query_agg.reset_index(), on='cookie_id', how='left')

    cat_text_cols = [
        'cat_platform_mode', 'cat_event_name_mode', 'cat_item_category_mode',
        'cat_item_location_mode', 'cat_seller_type_mode', 'text_user_agent',
        'text_search_queries'
    ]
    df_out[cat_text_cols] = df_out[cat_text_cols].fillna('__MISSING__').astype(str)
    numeric_cols = [c for c in df_out.columns if c not in cat_text_cols + ['cookie_id']]
    df_out[numeric_cols] = df_out[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df_out

In [69]:
add_ua_features(events)
add_temporal_features(events)

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

Xtr = extract_all_features(ev_tr, train)
Xte = extract_all_features(ev_te, test)

cat_feature_cols = [c for c in Xtr.columns if c.startswith('cat_')]
text_feature_cols = ['text_user_agent', 'text_search_queries']
catboost_feature_cols = [c for c in Xtr.columns if c != 'cookie_id']
feature_cols = [c for c in catboost_feature_cols if c not in cat_feature_cols + text_feature_cols]
ytr = train.target.values

print(f"Числовых признаков для LightGBM: {len(feature_cols)}")
print(f"Категориальных признаков для CatBoost: {len(cat_feature_cols)}")
print(f"Текстовых признаков для CatBoost: {len(text_feature_cols)}")

numeric_Xtr = Xtr[feature_cols].select_dtypes(include=[np.number])

Числовых признаков для LightGBM: 178
Категориальных признаков для CatBoost: 5
Текстовых признаков для CatBoost: 2


In [70]:
print('Всего признаков в Xtr/Xte:', len(Xtr.columns) - 1)
print('LightGBM получает:', len(feature_cols), 'числовых признаков')
print('CatBoost получает:', len(catboost_feature_cols), 'признаков =', len(cat_feature_cols), 'cat +', len(text_feature_cols), 'text +', len(feature_cols), 'numeric')
print('Неиспользуемые моделью колонки:', sorted(set(Xtr.columns) - {'cookie_id'} - set(catboost_feature_cols)))

Всего признаков в Xtr/Xte: 185
LightGBM получает: 178 числовых признаков
CatBoost получает: 185 признаков = 5 cat + 2 text + 178 numeric
Неиспользуемые моделью колонки: []


## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [71]:
import lightgbm as lgb
from metric import precision_at_recall

is_valid = train.window_start_ts.ge('2026-04-17').values

X_train_fold, y_train_fold = Xtr.loc[~is_valid, feature_cols], ytr[~is_valid]
X_val_fold, y_val_fold = Xtr.loc[is_valid, feature_cols], ytr[is_valid]
pos_weight = float((y_train_fold == 0).sum() / max((y_train_fold == 1).sum(), 1))
print('Доли ботов: train=', round(y_train_fold.mean(), 4), 'valid=', round(y_val_fold.mean(), 4), 'scale_pos_weight=', round(pos_weight, 3))

model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    force_col_wise=True,
    n_jobs=-1,
)

model.fit(X_train_fold, y_train_fold)

val_preds = model.predict_proba(X_val_fold)[:, 1]

regularized_model = lgb.LGBMClassifier(
    n_estimators=600, learning_rate=0.025, max_depth=5, num_leaves=15,
    min_child_samples=80, reg_lambda=5.0, colsample_bytree=0.75,
    random_state=42, force_col_wise=True, n_jobs=-1,
)
regularized_model.fit(X_train_fold, y_train_fold)
regularized_val_preds = regularized_model.predict_proba(X_val_fold)[:, 1]
lgb_val_preds = 0.5 * val_preds + 0.5 * regularized_val_preds
print('LGB baseline P@R0.7:', round(precision_at_recall(y_val_fold, val_preds), 4))
print('LGB regularized P@R0.7:', round(precision_at_recall(y_val_fold, regularized_val_preds), 4))
print('LGB 50/50 P@R0.7:', round(precision_at_recall(y_val_fold, lgb_val_preds), 4))
print('Константа:', round(y_val_fold.mean(), 4))

logreg_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.1, class_weight='balanced', max_iter=2000, solver='lbfgs', random_state=42)
)
logreg_model.fit(X_train_fold, y_train_fold)
logreg_val_preds = logreg_model.predict_proba(X_val_fold)[:, 1]
print('LogReg P@R0.7:', round(precision_at_recall(y_val_fold, logreg_val_preds), 4))

def ua_text_by_cookie(ev, meta):
    texts = ev.groupby('cookie_id')['user_agent'].agg(
        lambda s: ' '.join(pd.unique(s.fillna('').astype(str)))
    )
    return meta.cookie_id.map(texts).fillna('').to_numpy()

ua_text_train = ua_text_by_cookie(ev_tr, train)
ua_text_test = ua_text_by_cookie(ev_te, test)
ua_vectorizer = TfidfVectorizer(
    analyzer='char', ngram_range=(2, 5), min_df=2, max_features=15000,
    sublinear_tf=True, norm='l2'
)
X_ua_train = ua_vectorizer.fit_transform(ua_text_train[~is_valid])
X_ua_val = ua_vectorizer.transform(ua_text_train[is_valid])
ua_tfidf_model = LogisticRegression(C=2.0, class_weight='balanced', max_iter=2000, solver='liblinear', random_state=42)
ua_tfidf_model.fit(X_ua_train, y_train_fold)
ua_tfidf_val_preds = ua_tfidf_model.predict_proba(X_ua_val)[:, 1]
print('UA TF-IDF shape:', X_ua_train.shape)
print('UA TF-IDF LogReg P@R0.7:', round(precision_at_recall(y_val_fold, ua_tfidf_val_preds), 4))

rolling_rows = []
for fold_start in pd.to_datetime(['2026-04-11', '2026-04-14', '2026-04-17']):
    fold_end = fold_start + pd.Timedelta(days=3)
    fold_train = train.window_start_ts.lt(fold_start).to_numpy()
    fold_valid = train.window_start_ts.ge(fold_start).to_numpy() & train.window_start_ts.lt(fold_end).to_numpy()
    fold_pos_weight = float((ytr[fold_train] == 0).sum() / max((ytr[fold_train] == 1).sum(), 1))
    for weighting, scale in [('none', 1.0), ('balanced', fold_pos_weight)]:
        fold_model = lgb.LGBMClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=7, num_leaves=31,
            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale,
            random_state=42, force_col_wise=True, verbosity=-1, n_jobs=-1,
        )
        fold_model.fit(Xtr.loc[fold_train, feature_cols], ytr[fold_train])
        fold_pred = fold_model.predict_proba(Xtr.loc[fold_valid, feature_cols])[:, 1]
        rolling_rows.append({
            'fold_start': fold_start.date(), 'weighting': weighting,
            'n_valid': int(fold_valid.sum()), 'n_bots': int(ytr[fold_valid].sum()),
            'p_at_r70': precision_at_recall(ytr[fold_valid], fold_pred),
        })
rolling_scores = pd.DataFrame(rolling_rows)
display(rolling_scores.round(4))
display(rolling_scores.groupby('weighting').p_at_r70.agg(['mean', 'std', 'min']).round(4))

Доли ботов: train= 0.0809 valid= 0.082 scale_pos_weight= 11.368
LGB baseline P@R0.7: 0.5234
LGB regularized P@R0.7: 0.5305
LGB 50/50 P@R0.7: 0.5459
Константа: 0.082
LogReg P@R0.7: 0.3111
UA TF-IDF shape: (9140, 2238)
UA TF-IDF LogReg P@R0.7: 0.126


,fold_start,weighting,n_valid,n_bots,p_at_r70
0,2026-04-11,none,2556,199,0.3933
1,2026-04-11,balanced,2556,199,0.3701
2,2026-04-14,none,2367,195,0.4241
3,2026-04-14,balanced,2367,195,0.3881
4,2026-04-17,none,1951,160,0.5234
5,2026-04-17,balanced,1951,160,0.5308


,mean,std,min
weighting,,,
balanced,0.4297,0.0881,0.3701
none,0.4469,0.0680,0.3933


## CatBoost: категории и текстовые признаки

В LightGBM выше используются только числовые агрегаты. CatBoost получает mode-категории как `cat_features`, а User-Agent и объединённые поисковые запросы — как `text_features`.

In [72]:
cat_indices = [catboost_feature_cols.index(c) for c in cat_feature_cols]
text_indices = [catboost_feature_cols.index(c) for c in text_feature_cols]

X_train_cb = Xtr.loc[~is_valid, catboost_feature_cols].copy()
X_val_cb = Xtr.loc[is_valid, catboost_feature_cols].copy()
X_train_cb[cat_feature_cols + text_feature_cols] = X_train_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)
X_val_cb[cat_feature_cols + text_feature_cols] = X_val_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)

cat_model = CatBoostClassifier(
    iterations=2000,
    depth=8,
    learning_rate=0.01,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    l2_leaf_reg=3,
    thread_count=-1,
    allow_writing_files=False,
    verbose=100
)

cat_model.fit(
    X_train_cb, y_train_fold,
    cat_features=cat_indices,
    text_features=text_indices,
    eval_set=(X_val_cb, y_val_fold),
    use_best_model=True,
    early_stopping_rounds=150,
    verbose=100
)
cat_val_preds = cat_model.predict_proba(X_val_cb)[:, 1]
print('CatBoost best_iteration:', cat_model.get_best_iteration())
print('CatBoost P@R0.7:', round(precision_at_recall(y_val_fold, cat_val_preds), 4))

0:	test: 0.6875401	best: 0.6875401 (0)	total: 149ms	remaining: 4m 57s
100:	test: 0.8660001	best: 0.8662339 (98)	total: 7.98s	remaining: 2m 30s
200:	test: 0.8794005	best: 0.8795819 (186)	total: 15.8s	remaining: 2m 21s
300:	test: 0.8831937	best: 0.8832880 (299)	total: 23.7s	remaining: 2m 13s
400:	test: 0.8837486	best: 0.8845722 (366)	total: 31.5s	remaining: 2m 5s
500:	test: 0.8850154	best: 0.8850154 (500)	total: 40.5s	remaining: 2m 1s
600:	test: 0.8857796	best: 0.8858703 (586)	total: 48.4s	remaining: 1m 52s
700:	test: 0.8868090	best: 0.8871580 (666)	total: 55.8s	remaining: 1m 43s
800:	test: 0.8869033	best: 0.8872662 (757)	total: 1m 3s	remaining: 1m 34s
900:	test: 0.8872592	best: 0.8872662 (757)	total: 1m 10s	remaining: 1m 26s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.8872661921
bestIteration = 757

Shrink model to first 758 iterations.
CatBoost best_iteration: 757
CatBoost P@R0.7: 0.4934


In [73]:
def rank01(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()

blend_results = []
for cat_weight in np.linspace(0, 1, 11):
    for lr_weight in np.linspace(0, 1 - cat_weight, 11):
        for ua_weight in np.linspace(0, 1 - cat_weight - lr_weight, 11):
            lgb_weight = 1 - cat_weight - lr_weight - ua_weight
            raw_score = (lgb_weight * lgb_val_preds + cat_weight * cat_val_preds + lr_weight * logreg_val_preds + ua_weight * ua_tfidf_val_preds)
            rank_score = (lgb_weight * rank01(lgb_val_preds) + cat_weight * rank01(cat_val_preds) + lr_weight * rank01(logreg_val_preds) + ua_weight * rank01(ua_tfidf_val_preds))
            blend_results.append({
                'lgb_weight': lgb_weight, 'cat_weight': cat_weight, 'lr_weight': lr_weight, 'ua_weight': ua_weight,
                'raw_p_at_r70': precision_at_recall(y_val_fold, raw_score),
                'rank_p_at_r70': precision_at_recall(y_val_fold, rank_score),
            })


blend_results = pd.DataFrame(blend_results)
display(blend_results.sort_values('raw_p_at_r70', ascending=False).head(15).round(4))
best_raw = blend_results.loc[blend_results.raw_p_at_r70.idxmax()]
best_rank = blend_results.loc[blend_results.rank_p_at_r70.idxmax()]
if best_rank.rank_p_at_r70 > best_raw.raw_p_at_r70:
    ensemble_mode = 'rank'
    ensemble_weights = best_rank[['lgb_weight', 'cat_weight', 'lr_weight', 'ua_weight']].astype(float).to_dict()
    ensemble_val_score = float(best_rank.rank_p_at_r70)
else:
    ensemble_mode = 'raw'
    ensemble_weights = best_raw[['lgb_weight', 'cat_weight', 'lr_weight', 'ua_weight']].astype(float).to_dict()
    ensemble_val_score = float(best_raw.raw_p_at_r70)
print(f'Лучший режим: {ensemble_mode}, weights={ensemble_weights}, P@R0.7={ensemble_val_score:.4f}')

Path('artifacts').mkdir(exist_ok=True)
if ensemble_mode == 'raw':
    selected_ensemble_score = (ensemble_weights['lgb_weight'] * lgb_val_preds + ensemble_weights['cat_weight'] * cat_val_preds + ensemble_weights['lr_weight'] * logreg_val_preds + ensemble_weights['ua_weight'] * ua_tfidf_val_preds)
else:
    selected_ensemble_score = (ensemble_weights['lgb_weight'] * rank01(lgb_val_preds) + ensemble_weights['cat_weight'] * rank01(cat_val_preds) + ensemble_weights['lr_weight'] * rank01(logreg_val_preds) + ensemble_weights['ua_weight'] * rank01(ua_tfidf_val_preds))
pd.DataFrame({
    'cookie_id': train.loc[is_valid, 'cookie_id'].to_numpy(), 'target': y_val_fold,
    'lgb_score': lgb_val_preds, 'cat_score': cat_val_preds, 'lr_score': logreg_val_preds, 'ua_tfidf_score': ua_tfidf_val_preds,
    'ensemble_score': selected_ensemble_score,
}).to_csv('artifacts/validation_predictions.csv', index=False)

,lgb_weight,cat_weight,lr_weight,ua_weight,raw_p_at_r70,rank_p_at_r70
2,0.800,0.0,0.00,0.200,0.5803,0.4871
754,0.128,0.6,0.08,0.192,0.5714,0.5022
123,0.720,0.1,0.00,0.180,0.5685,0.5091
499,0.324,0.4,0.06,0.216,0.5657,0.4828
743,0.144,0.6,0.04,0.216,0.5657,0.4850
753,0.160,0.6,0.08,0.160,0.5628,0.5209
731,0.200,0.6,0.00,0.200,0.5628,0.5045
620,0.270,0.5,0.05,0.180,0.5628,0.5157
632,0.200,0.5,0.10,0.200,0.5628,0.4934
742,0.180,0.6,0.04,0.180,0.5628,0.5114


Лучший режим: rank, weights={'lgb_weight': 0.45, 'cat_weight': 0.5, 'lr_weight': 0.0, 'ua_weight': 0.05}, P@R0.7=0.5895


## Сабмит

In [74]:
model.fit(Xtr[feature_cols], ytr)
baseline_test_preds = model.predict_proba(Xte[feature_cols])[:, 1]
regularized_model.fit(Xtr[feature_cols], ytr)
regularized_test_preds = regularized_model.predict_proba(Xte[feature_cols])[:, 1]
lgb_test_preds = 0.5 * baseline_test_preds + 0.5 * regularized_test_preds

cat_iterations = max(1, cat_model.get_best_iteration() + 1)
X_full_cb = Xtr[catboost_feature_cols].copy()
X_test_cb = Xte[catboost_feature_cols].copy()
X_full_cb[cat_feature_cols + text_feature_cols] = X_full_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)
X_test_cb[cat_feature_cols + text_feature_cols] = X_test_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)

cat_model_full = CatBoostClassifier(
    iterations=cat_iterations,
    depth=8,
    learning_rate=0.01,
    loss_function='Logloss',
    random_seed=42,
    l2_leaf_reg=3,
    thread_count=-1,
    allow_writing_files=False,
    verbose=100
)
cat_model_full.fit(X_full_cb, ytr, cat_features=cat_indices, text_features=text_indices, verbose=100)
cat_test_preds = cat_model_full.predict_proba(X_test_cb)[:, 1]

logreg_model.fit(Xtr[feature_cols], ytr)
lr_test_preds = logreg_model.predict_proba(Xte[feature_cols])[:, 1]

ua_vectorizer_full = TfidfVectorizer(
    analyzer='char', ngram_range=(2, 5), min_df=2, max_features=15000,
    sublinear_tf=True, norm='l2'
)
X_ua_full = ua_vectorizer_full.fit_transform(ua_text_train)
X_ua_test = ua_vectorizer_full.transform(ua_text_test)
ua_tfidf_model_full = LogisticRegression(
    C=2.0, class_weight='balanced', max_iter=2000, solver='liblinear', random_state=42
)
ua_tfidf_model_full.fit(X_ua_full, ytr)
ua_tfidf_test_preds = ua_tfidf_model_full.predict_proba(X_ua_test)[:, 1]
print('UA TF-IDF full shape:', X_ua_full.shape)

if ensemble_mode == 'rank':
    ensemble_score = (ensemble_weights['lgb_weight'] * rank01(lgb_test_preds) + ensemble_weights['cat_weight'] * rank01(cat_test_preds) + ensemble_weights['lr_weight'] * rank01(lr_test_preds) + ensemble_weights['ua_weight'] * rank01(ua_tfidf_test_preds))
else:
    ensemble_score = (ensemble_weights['lgb_weight'] * lgb_test_preds + ensemble_weights['cat_weight'] * cat_test_preds + ensemble_weights['lr_weight'] * lr_test_preds + ensemble_weights['ua_weight'] * ua_tfidf_test_preds)

sub = pd.DataFrame({'cookie_id': Xte.cookie_id, 'score': ensemble_score})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
print(f'Финальный ансамбль: {ensemble_mode}, weights={ensemble_weights}, CatBoost iterations={cat_iterations}')
sub.head()

0:	learn: 0.6810723	total: 73.9ms	remaining: 55.9s
100:	learn: 0.2367169	total: 8.13s	remaining: 52.9s
200:	learn: 0.1698556	total: 16s	remaining: 44.4s
300:	learn: 0.1491696	total: 23.8s	remaining: 36.2s
400:	learn: 0.1403988	total: 32.3s	remaining: 28.7s
500:	learn: 0.1339834	total: 40.3s	remaining: 20.7s
600:	learn: 0.1294517	total: 48.5s	remaining: 12.7s
700:	learn: 0.1260863	total: 56.3s	remaining: 4.58s
757:	learn: 0.1240291	total: 1m	remaining: 0us
UA TF-IDF full shape: (11091, 2238)
Финальный ансамбль: rank, weights={'lgb_weight': 0.45, 'cat_weight': 0.5, 'lr_weight': 0.0, 'ua_weight': 0.05}, CatBoost iterations=758


,cookie_id,score
0,ck_315fb710a0e371e7,0.120615
1,ck_a76ee3b3e3e522fd,0.841378
2,ck_94c9a4d382689e82,0.373095
3,ck_8eaf9509ad9462a0,0.323936
4,ck_9a88a5a989cb5bc6,0.198931


## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.

In [75]:
importances_df = cat_model_full.get_feature_importance(prettified=True)
print(importances_df)

                 Feature Id  Importances
0     item_category_nunique     9.433227
1           gap_le_30s_mean     9.047755
2     item_location_nunique     5.819526
3             time_diff_q75     4.973734
4           text_user_agent     4.443703
..                      ...          ...
180             has_url_max     0.001952
181       is_known_tool_max     0.000000
182         is_headless_max     0.000000
183  ua_device_conflict_max     0.000000
184  cat_item_category_mode     0.000000

[185 rows x 2 columns]


In [76]:
import pandas as pd

importances = model.feature_importances_
feature_names = model.feature_name_

df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(df)

                  feature  importance
0         cookie_age_days         410
4   item_location_nunique         380
26       time_diff_median         363
30        gap_le_30s_mean         349
46          time_diff_q75         343
..                    ...         ...
15      is_app_header_max           1
69          hour_share_05           1
64          hour_share_00           0
13            has_url_max           0
54          session_count           0

[178 rows x 2 columns]


## SHAP-разбор ошибок

Отдельный LightGBM обучается только на train-части временного сплита. Встроенный TreeSHAP показывает, какие признаки толкают validation-cookie к боту (+) или человеку (-).

In [77]:
shap_model = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.03, max_depth=7, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    force_col_wise=True, n_jobs=-1, verbosity=-1,
)
shap_model.fit(X_train_fold, y_train_fold)
shap_raw = shap_model.predict(X_val_fold, pred_contrib=True)
shap_values = pd.DataFrame(shap_raw[:, :-1], columns=feature_cols)

order = np.argsort(-selected_ensemble_score, kind='mergesort')
y_sorted, score_sorted = y_val_fold[order].astype(int), selected_ensemble_score[order]
precision = np.cumsum(y_sorted) / np.arange(1, len(y_sorted) + 1)
recall = np.cumsum(y_sorted) / y_sorted.sum()
ends = np.r_[score_sorted[1:] != score_sorted[:-1], True]
valid_points = np.flatnonzero(ends & (recall >= 0.70))
shap_threshold = score_sorted[valid_points[np.argmax(precision[valid_points])]]

shap_errors = pd.DataFrame({
    'cookie_id': train.loc[is_valid, 'cookie_id'].to_numpy(),
    'target': y_val_fold, 'ensemble_score': selected_ensemble_score,
})
shap_errors['pred_bot'] = shap_errors.ensemble_score >= shap_threshold
shap_errors['error_type'] = np.select(
    [(shap_errors.target.eq(1) & ~shap_errors.pred_bot),
     (shap_errors.target.eq(0) & shap_errors.pred_bot),
     (shap_errors.target.eq(1) & shap_errors.pred_bot)],
    ['FN: bot missed', 'FP: human flagged', 'TP: bot caught'],
    default='TN: human passed',
)
shap_errors = shap_errors.join(shap_values)

global_shap = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': shap_values.abs().mean().to_numpy()})
global_shap = global_shap.sort_values('mean_abs_shap', ascending=False)
display(global_shap.head(25))

shap_by_error = shap_errors.groupby('error_type')[feature_cols].mean().T
display(shap_by_error.loc[global_shap.feature.head(25)].round(3))
display(shap_errors.error_type.value_counts().rename('cookies').to_frame())

for error_type in ['FP: human flagged', 'FN: bot missed']:
    print(f'\n{error_type}')
    examples = shap_errors[shap_errors.error_type.eq(error_type)].sort_values('ensemble_score', ascending=False).head(3)
    for _, row in examples.iterrows():
        print(f"cookie={row.cookie_id}, score={row.ensemble_score:.4f}")
        display(row[feature_cols].sort_values(key=np.abs, ascending=False).head(10).rename('shap_contribution').to_frame())

,feature,mean_abs_shap
3,item_category_nunique,0.481744
4,item_location_nunique,0.429742
26,time_diff_median,0.284103
30,gap_le_30s_mean,0.256693
2,item_id_nunique,0.243536
7,search_page_mean,0.209610
46,time_diff_q75,0.171752
0,cookie_age_days,0.152658
58,freq_favorite_add,0.141553
61,freq_photo_swipe,0.132049


error_type,FN: bot missed,FP: human flagged,TN: human passed,TP: bot caught
item_category_nunique,0.146,0.466,-0.051,0.444
item_location_nunique,0.009,0.382,-0.054,0.481
time_diff_median,0.023,0.322,-0.023,0.346
gap_le_30s_mean,-0.064,0.668,-0.133,1.368
item_id_nunique,0.034,0.193,-0.004,0.265
search_page_mean,0.053,0.180,-0.022,0.217
time_diff_q75,0.012,0.369,-0.048,0.411
cookie_age_days,0.043,0.101,-0.010,0.145
freq_favorite_add,0.017,0.075,-0.016,0.108
freq_photo_swipe,-0.007,0.061,-0.005,0.143


,cookies
error_type,
TN: human passed,1713
TP: bot caught,112
FP: human flagged,78
FN: bot missed,48



FP: human flagged
cookie=ck_e9ed95edb6b7bbf0, score=0.9795


,shap_contribution
gap_le_30s_mean,3.387625
item_category_nunique,0.623264
item_location_nunique,0.438151
time_diff_q75,0.432693
item_id_nunique,0.3307
cookie_age_days,0.29306
time_diff_median,0.242197
freq_favorite_add,0.170238
freq_photo_swipe,0.156249
search_query_nunique,0.150964


...

,shap_contribution
gap_le_30s_mean,2.246673
item_category_nunique,0.511544
time_diff_q75,0.461005
time_diff_median,0.330073
ua_length_std,-0.286368
item_id_nunique,-0.275601
cookie_age_days,-0.266828
time_diff_q25,0.25869
freq_seller_page_view,0.182887
search_page_mean,-0.176449


cookie=ck_3fbca87f98c2466f, score=0.8698


,shap_contribution
freq_contact_message_sent,0.645511
cookie_age_days,0.542611
search_page_mean,0.5363
time_diff_q10,0.474354
hour_diff_lag2_15,-0.473923
time_diff_median,-0.384324
item_category_nunique,0.354772
freq_photo_swipe,0.254278
time_diff_q75,-0.214543
time_diff_q25,-0.160504


cookie=ck_906eb72b7c24f458, score=0.8666


,shap_contribution
item_category_nunique,0.724359
freq_seller_page_view,0.5398
search_page_mean,0.406548
cookie_age_days,0.374078
item_location_nunique,0.371555
time_diff_q25,-0.275128
gap_le_30s_mean,-0.214757
time_diff_q75,-0.17101
hour_diff_lag1_07,-0.153641
digit_ratio_mean,0.148578
